# Training recurrent classifiers on IMDb

We will train a plain RNN, a bidirectional LSTM, and a GRU on the same movie reviews.
Start with a CPU runtime and run the cells in order. The first data run downloads
the two labelled splits of the public [Stanford IMDb dataset](https://huggingface.co/datasets/stanfordnlp/imdb)
(about 42 MB total). The models start with random weights and train from scratch.

## Setting up the CPU runtime

A standard Colab CPU runtime already provides PyTorch. In local Jupyter, install
[PyTorch for your platform](https://pytorch.org/get-started/locally/) in the notebook's
environment first. The setup cell installs the libraries for retrieving and reading the
dataset, building a tokenizer, and drawing the figures.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Install PyTorch in this notebook's environment, then rerun setup.")

packages = ["huggingface-hub", "pyarrow", "tokenizers", "matplotlib"]
missing = [
    package for package in packages
    if importlib.util.find_spec(package.replace("-", "_")) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

## Loading and splitting the reviews

Stanford supplies separate groups for training and testing. We draw balanced training
and validation subsets from the official training split, using distinct review indices.
We reserve a balanced sample of the official test split for the last evaluation.

In [ ]:
from collections import Counter
from html import unescape
from pathlib import Path
import os
import random
import re
from time import perf_counter

from huggingface_hub import hf_hub_download
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.normalizers import Lowercase
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader

SEED = 7
MAX_TOKENS = 128
BATCH_SIZE = 128
EPOCHS = 4
REVISION = "e6281661ce1c48d982bc483cf8a173c1bbeb5d31"
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
torch.set_num_threads(min(4, os.cpu_count() or 1))

def load_reviews(split):
    file = hf_hub_download(
        repo_id="stanfordnlp/imdb",
        filename=f"plain_text/{split}-00000-of-00001.parquet",
        repo_type="dataset",
        revision=REVISION,
        token=False,
    )
    return pq.read_table(file, columns=["text", "label"]).to_pylist()

def shuffled_labels(reviews, seed):
    rng = random.Random(seed)
    by_label = {}
    for label in (0, 1):
        indices = [i for i, review in enumerate(reviews) if review["label"] == label]
        rng.shuffle(indices)
        by_label[label] = indices
    return by_label

train_source = load_reviews("train")
test_source = load_reviews("test")
pools = shuffled_labels(train_source, SEED)
train_ids = [i for group in pools.values() for i in group[:2000]]
val_ids = [i for group in pools.values() for i in group[2000:2400]]
assert not set(train_ids) & set(val_ids)

train = [train_source[i] for i in train_ids]
validation = [train_source[i] for i in val_ids]
for name, split in [("Train", train), ("Validation", validation)]:
    counts = Counter(review["label"] for review in split)
    print(f"{name}: {len(split)} reviews ({counts[0]} negative, {counts[1]} positive)")

## Tokenizing, truncating, and batching

A word-level tokenizer maps whole words and punctuation to integer IDs. We fit it on
training reviews, so words seen only in validation or testing cannot enter the vocabulary.
When a review exceeds 128 tokens, we keep its final 128 tokens. This is a choice about
which text the model can see. A short review keeps all of its tokens.

In [ ]:
PAD, UNKNOWN = "[PAD]", "[UNK]"
tokenizer = Tokenizer(WordLevel(unk_token=UNKNOWN))
tokenizer.normalizer = Lowercase()
tokenizer.pre_tokenizer = Whitespace()
br_tag = re.compile(r"<br\s*/?>", flags=re.IGNORECASE)

def clean(text):
    return br_tag.sub(" ", unescape(text))

trainer = WordLevelTrainer(vocab_size=12000, special_tokens=[PAD, UNKNOWN])
tokenizer.train_from_iterator((clean(row["text"]) for row in train), trainer)
pad_id, unk_id = tokenizer.token_to_id(PAD), tokenizer.token_to_id(UNKNOWN)
assert pad_id != unk_id

def encode(text):
    ids = tokenizer.encode(clean(text)).ids
    return torch.tensor(ids[-MAX_TOKENS:] or [unk_id], dtype=torch.long)

def make_examples(rows):
    return [(encode(row["text"]), row["label"]) for row in rows]

def collate(examples):
    sequences, labels = zip(*examples)
    lengths = torch.tensor([len(sequence) for sequence in sequences])
    tokens = pad_sequence(sequences, batch_first=True, padding_value=pad_id)
    return tokens, lengths, torch.tensor(labels)

train_examples = make_examples(train)
validation_loader = DataLoader(
    make_examples(validation), batch_size=BATCH_SIZE, collate_fn=collate,
)
lengths_before_cap = [len(tokenizer.encode(clean(row["text"])).ids) for row in train]
truncated = sum(length > MAX_TOKENS for length in lengths_before_cap)
print(f"Training vocabulary: {tokenizer.get_vocab_size():,} token IDs")
print(f"Reviews over {MAX_TOKENS} tokens: {truncated:,}/{len(train):,}")
tokens, lengths, targets = collate([
    (encode("Slow."), 0), (encode("I loved the ending."), 1),
])
print(f"Two examples: {tuple(tokens.shape)}, lengths={lengths.tolist()}")

fig, ax = plt.subplots(figsize=(7, 3.2), layout="constrained")
ax.hist(lengths_before_cap, bins=range(0, 805, 25), color="#297d85", label="Training reviews")
ax.axvline(MAX_TOKENS, color="#a7472a", linestyle="--", linewidth=2, label="128-token cap")
ax.set(xlim=(0, 800), xlabel="Tokens per review before truncation", ylabel="Reviews",
       title="How much of each review can we read?")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
fig.savefig(OUTPUT / "imdb_review_lengths.png", dpi=240)
plt.show()

## Building the recurrent classifier

We embed the token IDs as trainable vectors. Packing tells PyTorch which positions
contain real tokens, so the final state is taken at each review's own boundary.
A bidirectional LSTM yields one final hidden state for each direction. Concatenating them
gives the classifier information from both readings of the same complete review.

In [ ]:
class ReviewClassifier(nn.Module):
    def __init__(self, kind):
        super().__init__()
        width = 32
        self.embed = nn.Embedding(tokenizer.get_vocab_size(), width, padding_idx=pad_id)
        recurrent = {"RNN": nn.RNN, "BiLSTM": nn.LSTM, "GRU": nn.GRU}[kind]
        bidirectional = kind == "BiLSTM"
        self.kind = kind
        self.encoder = recurrent(width, width, batch_first=True, bidirectional=bidirectional)
        self.head = nn.Linear(2 * width if bidirectional else width, 2)

    def forward(self, tokens, lengths):
        vectors = self.embed(tokens)
        packed = pack_padded_sequence(
            vectors, lengths.cpu(), batch_first=True, enforce_sorted=False,
        )
        _, state = self.encoder(packed)
        if self.kind == "BiLSTM":
            hidden, cell = state
            features = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            features = state[-1]
        return self.head(features)


def check_padding(model):
    model.eval()
    short = (encode("Slow."), 0)
    longer = (encode("I loved the ending."), 1)
    alone, alone_lengths, _ = collate([short])
    mixed, mixed_lengths, _ = collate([short, longer])
    with torch.inference_mode():
        both_logits = model(mixed, mixed_lengths)
        assert both_logits.shape == (2, 2)
        torch.testing.assert_close(model(alone, alone_lengths)[0], both_logits[0])

## Training and choosing weights on validation data

Each run uses the same review indices, vocabulary, batches and epoch limit. We keep
each model's weights from the epoch with the lowest validation loss. Use the printed
loss to see what happened during training, then inspect the test results in the next cell.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    total_loss = correct = count = 0
    for tokens, lengths, labels in loader:
        logits = model(tokens, lengths)
        total_loss += loss_fn(logits, labels).item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += len(labels)
    return total_loss / count, correct / count

def fit(kind):
    torch.manual_seed(SEED)
    model = ReviewClassifier(kind)
    check_padding(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loader = DataLoader(
        train_examples, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
        generator=torch.Generator().manual_seed(SEED),
    )
    best_loss, best_weights = float("inf"), None
    history = []
    started = perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for tokens, lengths, labels in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(tokens, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(labels)
        train_loss /= len(loader.dataset)
        val_loss, val_accuracy = evaluate(model, validation_loader)
        history.append((train_loss, val_loss, val_accuracy))
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = {name: value.clone() for name, value in model.state_dict().items()}
        print(f"{kind:6s} epoch {epoch + 1}: train loss {train_loss:.3f}, "
              f"val loss {val_loss:.3f}, val accuracy {val_accuracy:.1%}")
    model.load_state_dict(best_weights)
    print(f"{kind:6s} trained in {perf_counter() - started:.1f} seconds")
    return model, history

results = {kind: fit(kind) for kind in ("RNN", "BiLSTM", "GRU")}

## Evaluating the selected checkpoints

The 1,200-review test sample has equal numbers of positive and negative reviews.
A classifier that always picks one label gets 50% accuracy on this sample. The table
reports this small experiment, which uses only part of IMDb and one training setup.

In [ ]:
test_pools = shuffled_labels(test_source, SEED + 1)
test_ids = [i for group in test_pools.values() for i in group[:600]]
test_loader = DataLoader(
    make_examples([test_source[i] for i in test_ids]),
    batch_size=BATCH_SIZE, collate_fn=collate,
)

print(f"Test: {len(test_ids):,} reviews (600 negative, 600 positive)")
print("Model   Parameters   Best epoch   Test loss   Test accuracy")
for kind, (model, history) in results.items():
    best_epoch = min(range(EPOCHS), key=lambda i: history[i][1]) + 1
    size = sum(parameter.numel() for parameter in model.parameters())
    loss, accuracy = evaluate(model, test_loader)
    print(f"{kind:6s}  {size:10,d}  {best_epoch:11d}  {loss:10.3f}  {accuracy:13.1%}")

fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")
colors = {"RNN": "#a7472a", "BiLSTM": "#246f7b", "GRU": "#65518b"}
markers = {"RNN": "o", "BiLSTM": "s", "GRU": "^"}
for kind, (_, history) in results.items():
    losses = [row[1] for row in history]
    ax.plot(range(1, EPOCHS + 1), losses, label=kind, marker=markers[kind],
            color=colors[kind], linewidth=2)
ax.set(xlabel="Epoch", ylabel="Validation loss", title="Choosing weights by validation loss")
ax.set_xticks(range(1, EPOCHS + 1))
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
fig.savefig(OUTPUT / "imdb_validation_loss.png", dpi=240)
plt.show()

## Changing the order of a review

These two sentences use the same token multiset, including punctuation, but reverse
the order of the two descriptions. A change in score shows sensitivity to this pair.
Correct handling of contrast across other reviews would need a separate evaluation.

In [ ]:
examples = [
    "The film was dull at first, but the ending was wonderful.",
    "The ending was wonderful, but the film was dull at first.",
]
encoded = [encode(text) for text in examples]
assert Counter(encoded[0].tolist()) == Counter(encoded[1].tolist())
tokens, lengths, _ = collate([(sequence, 0) for sequence in encoded])
for text in examples:
    print(text)
with torch.inference_mode():
    for kind, (model, _) in results.items():
        model.eval()
        scores = model(tokens, lengths).softmax(dim=1)[:, 1]
        print(f"{kind:6s} positive scores: {scores[0]:.3f} and {scores[1]:.3f}")